In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
import re
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
DATA_DIR = '/kaggle/input/bdh-hackz'
OUTPUT_DIR = '/kaggle/working'
BOOKS_DIR = f'{DATA_DIR}/Books'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"Random seed set to {SEED} for reproducibility")
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Train: {len(train_df)}, Test: {len(test_df)}")
class BDHState(nn.Module):
    def __init__(self, state_dim=256, sparsity=0.15):
        super().__init__()
        self.state_dim = state_dim
        self.sparsity = sparsity
        self.register_buffer("state", torch.zeros(state_dim))
        
        self.update_gate = nn.Sequential(
            nn.Linear(state_dim * 2, state_dim),
            nn.Sigmoid()
        )
        self.candidate_layer = nn.Sequential(
            nn.Linear(state_dim * 2, state_dim),
            nn.Tanh()
        )
        
        self.importance_scorer = nn.Sequential(
            nn.Linear(state_dim * 2, state_dim),
            nn.Sigmoid()
        )
        
        self.layer_norm = nn.LayerNorm(state_dim)
        
        self.update_history = []
        self.sparsity_metrics = []
        
        self.beliefs = {
            "alive_status": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
            "birth_year": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
            "death_year": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
            "home_location": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
            "family_status": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
            "occupation": {"value": None, "confidence": 0.0, "locked": False, "evidence": []},
        }
        self.belief_history = []
        
    def reset_state(self):
        self.state = torch.zeros(self.state_dim, device=self.state.device)
        self.update_history = []
        self.sparsity_metrics = []
        for key in self.beliefs:
            self.beliefs[key] = {"value": None, "confidence": 0.0, "locked": False, "evidence": []}
        self.belief_history = []
        
    def update(self, chunk_embedding):
        combined = torch.cat([self.state, chunk_embedding], dim=-1)
        
        importance = self.importance_scorer(combined)
        
        k = max(1, int(self.state_dim * self.sparsity))
        topk_values, topk_indices = torch.topk(importance, k)
        
        sparse_mask = torch.zeros_like(self.state)
        sparse_mask[topk_indices] = 1.0
        
        z = self.update_gate(combined)
        h_tilde = self.candidate_layer(combined)
        candidate_update = (1 - z) * self.state + z * h_tilde
        
        self.state = sparse_mask * candidate_update + (1 - sparse_mask) * self.state
        self.state = self.layer_norm(self.state)
        
        self.update_history.append(topk_indices.detach().cpu().numpy())
        self.sparsity_metrics.append({
            'k': k,
            'updated_dims': topk_indices.tolist() if hasattr(topk_indices, 'tolist') else list(topk_indices.detach().cpu().numpy()),
            'max_importance': topk_values.max().item() if len(topk_values) > 0 else 0
        })
        
        return self.state
    
    def update_belief(self, belief_key, value, evidence_text, strength=0.15):
        if belief_key not in self.beliefs:
            return False
        
        belief = self.beliefs[belief_key]
        
        if belief["value"] is not None and belief["value"] != value:
            if "contradictions" not in belief:
                belief["contradictions"] = []
            belief["contradictions"].append({
                "expected": belief["value"],
                "found": value,
                "evidence": evidence_text[:200] if len(evidence_text) > 200 else evidence_text,
                "chunk_idx": len(self.sparsity_metrics)
            })
            
            if belief["locked"]:
                belief["major_contradiction"] = True
        
        if belief["locked"]:
            if belief["value"] != value and strength < belief["confidence"]:
                return False
        
        if belief["value"] is None:
            belief["value"] = value
            belief["confidence"] = strength
        elif belief["value"] == value:
            belief["confidence"] = min(1.0, belief["confidence"] + strength)
        else:
            if strength > belief["confidence"]:
                belief["value"] = value
                belief["confidence"] = strength
                belief["evidence"] = []
        
        belief["evidence"].append(evidence_text[:200] if len(evidence_text) > 200 else evidence_text)
        
        if belief["confidence"] >= 0.85:
            belief["locked"] = True
        
        self.belief_history.append({
            "belief": belief_key,
            "value": value,
            "confidence": belief["confidence"],
            "locked": belief["locked"],
            "chunk_idx": len(self.sparsity_metrics)
        })
        
        return True
    
    def extract_and_update_beliefs(self, chunk_text):
        import re
        text_lower = chunk_text.lower()
        
        negation_words = ['not', 'never', 'no', "wasn't", "weren't", "didn't", "isn't", 'neither', 'nor']
        
        def is_negated(pattern, text):
            if pattern not in text:
                return True
            idx = text.index(pattern)
            context = text[max(0, idx-30):idx]
            return any(neg in context for neg in negation_words)
        
        death_patterns = ['died', 'death', 'killed', 'murdered', 'deceased', 'passed away']
        for pattern in death_patterns:
            if pattern in text_lower:
                if not is_negated(pattern, text_lower):
                    self.update_belief("alive_status", False, chunk_text, strength=0.2)
                    break
        
        alive_patterns = ['alive', 'living', 'survived', 'lives']
        for pattern in alive_patterns:
            if pattern in text_lower:
                if not is_negated(pattern, text_lower):
                    self.update_belief("alive_status", True, chunk_text, strength=0.15)
                    break
        
        birth_match = re.search(r'born\s+(?:in\s+)?(\d{4})', text_lower)
        if birth_match:
            self.update_belief("birth_year", int(birth_match.group(1)), chunk_text, strength=0.25)
        
        death_match = re.search(r'died\s+(?:in\s+)?(\d{4})', text_lower)
        if death_match:
            self.update_belief("death_year", int(death_match.group(1)), chunk_text, strength=0.25)
        
        if 'orphan' in text_lower:
            if is_negated('orphan', text_lower):
                self.update_belief("family_status", "has_parents", chunk_text, strength=0.25)
            else:
                self.update_belief("family_status", "orphan", chunk_text, strength=0.3)
        elif any(p in text_lower for p in ['father', 'mother', 'parent']):
            if not any(is_negated(p, text_lower) for p in ['father', 'mother', 'parent'] if p in text_lower):
                self.update_belief("family_status", "has_parents", chunk_text, strength=0.15)
    
    def get_beliefs_summary(self):
        summary = {}
        for key, belief in self.beliefs.items():
            if belief["value"] is not None:
                summary[key] = {
                    "value": belief["value"],
                    "confidence": round(belief["confidence"], 2),
                    "locked": belief["locked"],
                    "evidence_count": len(belief["evidence"])
                }
        return summary
    
    def get_belief_evolution(self):
        return self.belief_history
    
    def get_sparsity_stats(self):
        if not self.sparsity_metrics:
            return {"avg_updated": 0, "sparsity_rate": 0}
        
        total_updated = sum(m['k'] for m in self.sparsity_metrics)
        avg_updated = total_updated / len(self.sparsity_metrics)
        sparsity_rate = 1 - (avg_updated / self.state_dim)
        
        return {
            "avg_dimensions_updated": avg_updated,
            "sparsity_rate": sparsity_rate,
            "total_chunks_processed": len(self.sparsity_metrics),
            "percentage_sparse": f"{sparsity_rate * 100:.1f}%"
        }

class CausalPathChecker(nn.Module):
    def __init__(self, state_dim=256):
        super().__init__()
        
        self.path_validator = nn.Sequential(
            nn.Linear(state_dim * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
    def forward(self, backstory_state, early_novel_state, late_novel_state):
        combined = torch.cat([backstory_state, early_novel_state, late_novel_state], dim=-1)
        return self.path_validator(combined)

class IncrementalBeliefTracker:
    def __init__(self):
        self.belief_history = []
        self.current_belief = 0.5
        self.total_chunks = 0
        
    def reset(self):
        self.belief_history = []
        self.current_belief = 0.5
        self.total_chunks = 0
    
    def update(self, neural_signal, chunk_idx, chunk_text="", bdh_beliefs=None):
        confidence = neural_signal
        
        num_locked = 0
        num_contradictions = 0
        
        if bdh_beliefs is not None:
            num_locked = sum(1 for b in bdh_beliefs.values() if b.get("locked", False))
            
            num_contradictions = sum(
                len(b.get("contradictions", [])) 
                for b in bdh_beliefs.values()
            )
            
            confidence += num_locked * 0.02
            
            confidence -= num_contradictions * 0.1
            
            confidence = max(0.0, min(1.0, confidence))
        
        if self.total_chunks > 0:
            position_ratio = chunk_idx / self.total_chunks
        else:
            position_ratio = 0.5
        
        if position_ratio < 0.3:
            weight = 0.7
        elif position_ratio > 0.7:
            weight = 0.3
        else:
            weight = 0.5
        
        self.current_belief = weight * self.current_belief + (1 - weight) * confidence
        
        self.belief_history.append({
            'chunk_idx': chunk_idx,
            'confidence': round(self.current_belief, 4),
            'signal': round(neural_signal, 4),
            'position': round(position_ratio, 2),
            'num_locked_beliefs': num_locked,
            'num_contradictions': num_contradictions,
            'snippet': chunk_text[:100] if chunk_text else ""
        })
        
        return self.current_belief
    
    def set_total_chunks(self, total):
        self.total_chunks = total
    
    def get_belief_trajectory(self):
        return {
            'history': self.belief_history,
            'final_belief': self.current_belief,
            'total_updates': len(self.belief_history),
            'trajectory': [h['confidence'] for h in self.belief_history]
        }
    
    def get_summary(self):
        if not self.belief_history:
            return {"status": "no_data"}
        
        confidences = [h['confidence'] for h in self.belief_history]
        return {
            'initial': 0.5,
            'final': self.current_belief,
            'min': min(confidences),
            'max': max(confidences),
            'trend': 'increasing' if confidences[-1] > confidences[0] else 'decreasing',
            'total_updates': len(self.belief_history)
        }

class ChunkEncoder(nn.Module):
    def __init__(self, state_dim=256):
        super().__init__()
        self.encoder = SentenceTransformer('all-mpnet-base-v2')
        self.encoder_dim = 768
        self.projection = nn.Linear(self.encoder_dim, state_dim)
        self.bdh_state = BDHState(state_dim=state_dim)
        
        self.belief_tracker = IncrementalBeliefTracker()
        
    def reset(self):
        self.bdh_state.reset_state()
        self.belief_tracker.reset()
        
    def encode_text(self, text):
        with torch.no_grad():
            emb = self.encoder.encode(text, convert_to_tensor=True)
        emb = emb.clone().detach().requires_grad_(self.training)
        return self.projection(emb)
        
    def process_chunk(self, chunk_text):
        projected = self.encode_text(chunk_text)
        state = self.bdh_state.update(projected)
        self.bdh_state.extract_and_update_beliefs(chunk_text)
        return state

    def process_chunks_in_batch(self, chunks, device='cuda'):
        """
        Optimized batch processing:
        1. Batch encode all chunks (Parallel GPU usage)
        2. Sequential state update (Required for RNN nature)
        """
        if not chunks:
            return []
            
        # 1. Batch Encode
        with torch.no_grad():
            embeddings = self.encoder.encode(chunks, convert_to_tensor=True, device=device, batch_size=32)
        
        # 2. Sequential Update
        states = []
        projected_embs = self.projection(embeddings) # Batch projection
        
        for i, chunk_text in enumerate(chunks):
            # Update state with pre-computed embedding
            state = self.bdh_state.update(projected_embs[i])
            self.bdh_state.extract_and_update_beliefs(chunk_text)
            states.append(state.clone())
            
        return states
    
    def process_novel_with_tracking(self, chunks, classifier=None, backstory_emb=None):
        self.reset()
        self.belief_tracker.set_total_chunks(len(chunks))
        
        for idx, chunk in enumerate(chunks):
            state = self.process_chunk(chunk)
            
            if classifier is not None and backstory_emb is not None:
                with torch.no_grad():
                    output = classifier(state, backstory_emb, None)
                    confidence = torch.sigmoid(output).item()
                    self.belief_tracker.update(confidence, idx, chunk, 
                                               bdh_beliefs=self.bdh_state.beliefs)
        
        return {
            'final_state': self.bdh_state.state,
            'sparsity_stats': self.bdh_state.get_sparsity_stats(),
            'beliefs': self.bdh_state.get_beliefs_summary(),
            'belief_trajectory': self.belief_tracker.get_belief_trajectory(),
            'total_chunks_processed': len(chunks)
        }
UNIVERSAL_FEATURE_DIM = 12
class CausalCompatibilityClassifier(nn.Module):
    def __init__(self, state_dim=256, use_universal_features=True):
        super().__init__()
        self.use_universal_features = use_universal_features
        self.state_dim = state_dim
        
        self.causal_checker = CausalPathChecker(state_dim)
        
        self.feature_interact = nn.Linear(state_dim * 2, state_dim)
        
        if use_universal_features:
            self.universal_layer = nn.Sequential(
                nn.Linear(UNIVERSAL_FEATURE_DIM, 32),
                nn.ReLU(),
                nn.Dropout(0.4)
            )
            combined_dim = state_dim + 32 + 1
        else:
            self.universal_layer = None
            combined_dim = state_dim + 1
        
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.LayerNorm(128),  # FIX: BatchNorm crashes on batch_size=1
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.LayerNorm(64),   # FIX: BatchNorm crashes on batch_size=1
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    
    def forward(self, backstory_state, trajectory, universal_features=None):
        if isinstance(trajectory, tuple):
            early_novel, late_novel = trajectory
        else:
            early_states = [t['state'] for t in trajectory if t['position'] < 0.3]
            late_states = [t['state'] for t in trajectory if t['position'] > 0.7]
            
            if early_states:
                early_novel = torch.stack(early_states).mean(dim=0)
            else:
                early_novel = torch.zeros_like(backstory_state)
            
            if late_states:
                late_novel = torch.stack(late_states).mean(dim=0)
            else:
                late_novel = torch.zeros_like(backstory_state)
        
        early_novel = early_novel.to(backstory_state.device)
        late_novel = late_novel.to(backstory_state.device)
        
        if early_novel.dim() == 1:
            early_novel = early_novel.unsqueeze(0)
        if late_novel.dim() == 1:
            late_novel = late_novel.unsqueeze(0)
        
        causal_score = self.causal_checker(backstory_state, early_novel, late_novel)
        
        combined = torch.cat([backstory_state, late_novel], dim=-1)
        interacted = torch.relu(self.feature_interact(combined))
        
        if self.use_universal_features and universal_features is not None:
            universal_processed = self.universal_layer(universal_features)
            final_features = torch.cat([interacted, universal_processed, causal_score], dim=-1)
        else:
            final_features = torch.cat([interacted, causal_score], dim=-1)
        
        return self.classifier(final_features)

CompatibilityClassifier = CausalCompatibilityClassifier
def chunk_novel(text, chunk_size=200, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        if len(chunk) > 50:
            chunks.append(chunk)
    return chunks

def get_backstory_relevant_chunks(backstory, all_chunks, encoder, top_k=200, cached_embeddings=None):
    """
    BACKSTORY-BASED SELECTION: 
    1. Encode ALL novel chunks (or use cached)
    2. Compare each to backstory embedding
    3. Return TOP K most similar chunks
    """
    if not all_chunks:
        return []
    
    with torch.no_grad():
        backstory_emb = encoder.encoder.encode(backstory, convert_to_tensor=True, device=device)
        
        if cached_embeddings is not None:
            all_chunk_embs = cached_embeddings
        else:
            # Batch processing with safety limit
            all_chunk_embs = encoder.encoder.encode(all_chunks, convert_to_tensor=True, device=device, batch_size=32)
            
        similarities = util.cos_sim(backstory_emb, all_chunk_embs)[0]
        top_indices = torch.argsort(similarities, descending=True)[:top_k]
    
    selected_chunks = [all_chunks[i] for i in top_indices]
    # print(f"    Selected top {len(selected_chunks)} chunks...") # Removed spammy print
    return selected_chunks
_filename_cache = {}
def load_novel(book_name, books_dir):
    # 1. Fast path: Check specific file path
    path = os.path.join(books_dir, f"{book_name}.txt")
    if os.path.exists(path):
        return open(path, 'r', encoding='utf-8').read()
    
    # 2. Cached directory scan (Optimized)
    global _filename_cache
    if not _filename_cache:
        print("  ⚡ Building book filename cache...")
        for f in os.listdir(books_dir):
            if f.endswith('.txt'):
                _filename_cache[f.lower()] = f
    
    # 3. Cache lookup
    target_lower = f"{book_name.lower()}.txt"
    if target_lower in _filename_cache:
        return open(os.path.join(books_dir, _filename_cache[target_lower]), 'r', encoding='utf-8').read()
        
    # 4. Fallback search (rare)
    for f in os.listdir(books_dir):
        if book_name.lower() in f.lower():
            return open(os.path.join(books_dir, f), 'r', encoding='utf-8').read()
            
    print(f"WARNING: Book not found: {book_name}")
    return ""
def is_relevant_chunk(chunk, char_name):
    name_parts = char_name.split()
    for part in name_parts:
        if len(part) < 3:
            continue
        pattern = r'\b' + re.escape(part) + r'\b'
        if re.search(pattern, chunk, re.IGNORECASE):
            return True
    return False
def find_evidence_chunks(backstory, novel_chunks, chunk_embs, encoder, top_k=3):
    if not novel_chunks:
        return []
    with torch.no_grad():
        backstory_emb = encoder.encoder.encode(backstory, convert_to_tensor=True)
        similarities = util.cos_sim(backstory_emb, chunk_embs)[0]
        top_indices = torch.argsort(similarities, descending=True)[:top_k]
    evidence = []
    for idx in top_indices:
        evidence.append({
            'chunk': novel_chunks[idx],
            'similarity': similarities[idx].item()
        })
    return evidence
def rule_based_check(backstory, evidence_chunks):
    backstory_lower = backstory.lower()
    reasons = []
    vote = None
    death_words = ['died', 'death', 'killed', 'murdered', 'dead']
    alive_words = ['lived', 'survived', 'escaped', 'continued']
    backstory_mentions_death = any(w in backstory_lower for w in death_words)
    evidence_shows_alive = False
    for ev in evidence_chunks:
        if any(w in ev['chunk'].lower() for w in alive_words):
            evidence_shows_alive = True
            break
    if backstory_mentions_death and evidence_shows_alive:
        vote = 0
        reasons.append(f"Backstory mentions death, but novel shows character survived.")
    location_patterns = [
        (r'born in (\w+)', 'birthplace'),
        (r'lived in (\w+)', 'residence'),
        (r'from (\w+)', 'origin')
    ]
    for pattern, label in location_patterns:
        backstory_match = re.search(pattern, backstory_lower)
        if backstory_match:
            backstory_location = backstory_match.group(1)
            for ev in evidence_chunks:
                ev_match = re.search(pattern, ev['chunk'].lower())
                if ev_match:
                    ev_location = ev_match.group(1)
                    if backstory_location != ev_location:
                        vote = 0
                        reasons.append(f"Backstory says {label}='{backstory_location}', but novel says '{ev_location}'.")
                        break
    if len(backstory.split()) < 15 and vote is None:
        vote = 1
        reasons.append("Backstory is vague/short, likely plausible.")
    return vote, reasons
class UniversalFeatureExtractor:
    def __init__(self):
        self.birth_patterns = [
            re.compile(r'\b(?:born|birth)\s+(?:in\s+)?(\d{4})'),
            re.compile(r'\b(\d{1,2})\s*years?\s*old\b'),
            re.compile(r'\b(?:age|aged)\s+(\d{1,2})\b'),
        ]
        self.death_patterns = [
            re.compile(r'\b(?:died|death|killed|murdered|deceased|passed away)\b'),
        ]
        self.alive_patterns = [
            re.compile(r'\b(?:survived|lives|living|alive|escaped death)\b'),
        ]
        self.family_patterns = [
            re.compile(r'\b(?:father|mother|parent|brother|sister|sibling|son|daughter|child|orphan|widow)\b'),
            re.compile(r'\b(?:married|husband|wife|spouse|wedding)\b'),
            re.compile(r'\b(?:family|relatives|kin|ancestry|lineage)\b'),
        ]
        self.location_patterns = [
            re.compile(r'\b(?:born in|from|lived in|grew up in|native of)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)\b'),
        ]
        self.profession_patterns = [
            re.compile(r'\b(?:worked as|profession|occupation|job|career|employed as)\b'),
            re.compile(r'\b(?:doctor|lawyer|merchant|farmer|soldier|servant|governess|teacher|priest)\b'),
        ]
        self.personality_patterns = [
            re.compile(r'\b(?:brave|cowardly|kind|cruel|intelligent|foolish|honest|deceitful)\b'),
            re.compile(r'\b(?:proud|humble|generous|greedy|loyal|treacherous|passionate|cold)\b'),
        ]
        self.wealth_patterns = [
            re.compile(r'\b(?:rich|wealthy|poor|poverty|fortune|estate|inheritance|destitute)\b'),
            re.compile(r'\b(?:money|gold|debt|bankrupt|prosperous|impoverished)\b'),
        ]
        self.education_patterns = [
            re.compile(r'\b(?:educated|school|university|learned|illiterate|scholar|tutor)\b'),
        ]
        self.timeline_patterns = [
            re.compile(r'\b(?:before|after|during|when|while|until|since)\b.*\b(?:war|revolution|marriage|death|birth)\b'),
        ]
        self.negation_intensifiers = [
            re.compile(r'\b(?:never|not|no|none|nothing|nowhere|nobody|neither|cannot|impossible)\b'),
        ]
    def extract_features(self, backstory, evidence_chunks):
        backstory_lower = backstory.lower()
        evidence_text = ' '.join([ev['chunk'].lower() for ev in evidence_chunks]) if evidence_chunks else ''
        features = {}
        features['has_birth_claim'] = float(any(
            p.search(backstory_lower) for p in self.birth_patterns
        ))
        features['has_death_claim'] = float(any(
            p.search(backstory_lower) for p in self.death_patterns
        ))
        features['birth_contradiction'] = self._check_birth_contradiction(backstory_lower, evidence_text)
        features['death_contradiction'] = self._check_death_contradiction(backstory_lower, evidence_text)
        features['has_family_claim'] = float(any(
            p.search(backstory_lower) for p in self.family_patterns
        ))
        features['has_location_claim'] = float(any(
             p.search(backstory) for p in self.location_patterns # kept case sensitive
        ))
        features['has_profession_claim'] = float(any(
            p.search(backstory_lower) for p in self.profession_patterns
        ))
        features['has_personality_claim'] = float(any(
            p.search(backstory_lower) for p in self.personality_patterns
        ))
        features['has_wealth_claim'] = float(any(
            p.search(backstory_lower) for p in self.wealth_patterns
        ))
        negation_count = sum(
            len(p.findall(backstory_lower)) for p in self.negation_intensifiers
        )
        features['negation_intensity'] = min(1.0, negation_count / 5.0)
        word_count = len(backstory.split())
        features['backstory_specificity'] = min(1.0, word_count / 100.0)
        if evidence_chunks:
            avg_similarity = np.mean([ev['similarity'] for ev in evidence_chunks])
            features['evidence_match_score'] = float(avg_similarity)
        else:
            features['evidence_match_score'] = 0.5
        return features
    def _check_birth_contradiction(self, backstory, evidence):
        backstory_years = re.findall(r'\b(1[6-9]\d{2})\b', backstory)
        evidence_years = re.findall(r'\b(1[6-9]\d{2})\b', evidence)
        if backstory_years and evidence_years:
            backstory_set = set(backstory_years)
            evidence_set = set(evidence_years)
            if backstory_set and evidence_set and not backstory_set.intersection(evidence_set):
                return 0.7
        return 0.0
    def _check_death_contradiction(self, backstory, evidence):
        backstory_death = any(p.search(backstory) for p in self.death_patterns)
        backstory_alive = any(p.search(backstory) for p in self.alive_patterns)
        evidence_death = any(p.search(evidence) for p in self.death_patterns)
        evidence_alive = any(p.search(evidence) for p in self.alive_patterns)
        if backstory_death and evidence_alive:
            return 1.0
        if backstory_alive and evidence_death:
            return 0.8
        return 0.0
    def get_feature_vector(self, backstory, evidence_chunks):
        features = self.extract_features(backstory, evidence_chunks)
        feature_order = [
            'has_birth_claim', 'has_death_claim', 'birth_contradiction',
            'death_contradiction', 'has_family_claim', 'has_location_claim',
            'has_profession_claim', 'has_personality_claim', 'has_wealth_claim',
            'negation_intensity', 'backstory_specificity', 'evidence_match_score'
        ]
        return np.array([features[f] for f in feature_order], dtype=np.float32)
universal_extractor = UniversalFeatureExtractor()
def generate_rationale(backstory, evidence_chunks, nn_pred, nn_conf, rule_vote, rule_reasons, final_pred):
    if rule_vote is not None and rule_vote != nn_pred:
        if final_pred == rule_vote:
            return f"Rule-based override: {'; '.join(rule_reasons)}"
        else:
            return f"Model confidence ({nn_conf:.0%}) outweighed rule signals ({'; '.join(rule_reasons)})"
    if rule_vote is not None and rule_vote == nn_pred:
        return f"Model ({nn_conf:.0%}) and rules agree: {'; '.join(rule_reasons)}"
    pred_str = "consistent" if final_pred == 1 else "contradictory"
    if evidence_chunks:
        top_sim = evidence_chunks[0]['similarity']
        return f"Model predictive confidence {nn_conf:.0%} (with supporting text similarity {top_sim:.2f})"
    return f"Model prediction (confidence {nn_conf:.0%}) based on narrative consistency"
PARAPHRASE_RULES = [
    (r'(\d+) years old', r'at the age of \1'),
    (r'was born in (\d+)', r'came into the world in \1'),
    (r'grew up', 'was raised'),
    (r'as a child', 'during childhood'),
    (r'in his youth', 'when he was young'),
    (r'in her youth', 'when she was young'),
    (r'his father', 'his dad'),
    (r'her father', 'her dad'),
    (r'his mother', 'his mom'),
    (r'her mother', 'her mom'),
    (r'was an orphan', 'lost both parents'),
    (r'poor family', 'impoverished family'),
    (r'wealthy family', 'rich family'),
    (r'was brave', 'showed courage'),
    (r'was afraid', 'lived in fear'),
    (r'loved', 'was passionate about'),
    (r'hated', 'despised'),
    (r'skilled in', 'proficient at'),
    (r'traveled to', 'journeyed to'),
    (r'escaped from', 'fled from'),
    (r'was betrayed by', 'was deceived by'),
    (r'married', 'wed'),
]
NEGATION_RULES = [
    (r'\bwas\b', 'was never'),
    (r'\bhad\b', 'never had'),
    (r'\bloved\b', 'hated'),
    (r'\bfeared\b', 'never feared'),
    (r'\balways\b', 'never'),
    (r'\bknown for\b', 'not known for'),
    (r'\bskilled in\b', 'unskilled in'),
    (r'\bgrew up\b', 'did not grow up'),
]
EMPHASIS_ADDITIONS = {
    'brave': ' and courageous',
    'poor': ' and impoverished',
    'wealthy': ' and prosperous',
    'feared': ', deeply feared',
    'loved': ' and cherished',
    'skilled': ' and talented',
    'betrayed': ' and deceived',
}
def paraphrase_backstory(text):
    modified = text
    rules_to_apply = random.sample(PARAPHRASE_RULES, min(3, len(PARAPHRASE_RULES)))
    for pattern, replacement in rules_to_apply:
        modified = re.sub(pattern, replacement, modified, flags=re.IGNORECASE)
    return modified
def negate_backstory(text):
    modified = text
    for pattern, replacement in NEGATION_RULES:
        if re.search(pattern, modified, re.IGNORECASE):
            modified = re.sub(pattern, replacement, modified, count=1, flags=re.IGNORECASE)
            break
    return modified
def emphasize_keywords(text):
    modified = text
    for keyword, addition in EMPHASIS_ADDITIONS.items():
        if keyword.lower() in modified.lower():
            pattern = rf'({keyword})'
            modified = re.sub(pattern, rf'\1{addition}', modified, count=1, flags=re.IGNORECASE)
            break
    return modified
def create_augmented_data(df):
    augmented = []
    for _, row in df.iterrows():
        text = row['content']
        original_label = row['label']
        
        # FIX 2: Only ONE augmentation per sample (not 4) - reduces to ~2x increase
        if random.random() > 0.5:
            para_text = paraphrase_backstory(text)
            if para_text != text:
                augmented.append({
                    'id': f"{row['id']}_para",
                    'book_name': row['book_name'],
                    'char': row['char'],
                    'content': para_text,
                    'label': original_label
                })
        else:
            emph_text = emphasize_keywords(text)
            if emph_text != text:
                augmented.append({
                    'id': f"{row['id']}_emph",
                    'book_name': row['book_name'],
                    'char': row['char'],
                    'content': emph_text,
                    'label': original_label
                })
    
    if not augmented:
        print("⚠️ No augmentation applied (no patterns matched)")
        return df
    
    aug_df = pd.DataFrame(augmented)
    combined = pd.concat([df, aug_df], ignore_index=True)
    print(f"🔄 AUGMENTATION COMPLETE (2x safer):")
    print(f"   Original: {len(df)} samples")
    print(f"   Added: {len(augmented)} samples")
    print(f"   Total: {len(combined)} samples ({len(combined)/len(df):.1f}x increase)")
    return combined
# Removed dead code: precompute_book_states (unused)

def precompute_trajectories(df, encoder, books_dir, max_chunks=200, num_snapshots=10):
    print("\n🎬 PHASE 2: Pre-computing STATE TRAJECTORIES (for causal analysis)...")
    
    book_char_trajectories = {}
    encoder.eval()
    
    unique_pairs = df[['book_name', 'char']].drop_duplicates()
    print(f"Found {len(unique_pairs)} unique (book, character) pairs")
    
    # Sort by book_name to maximize cache hits
    unique_pairs = unique_pairs.sort_values('book_name')
    
    # Caching variables
    current_book_name = None
    current_novel_text = None
    current_chunks = None
    
    for idx, row in unique_pairs.iterrows():
        book_name = row['book_name']
        char_name = row['char']
        key = (book_name, char_name)
        
        try:
            # Check cache
            if book_name != current_book_name:
                novel_text = load_novel(book_name, books_dir)
                if not novel_text:
                    book_char_trajectories[key] = None
                    continue
                chunks = chunk_novel(novel_text)
                
                # Update cache
                current_book_name = book_name
                current_novel_text = novel_text
                current_chunks = chunks
            else:
                chunks = current_chunks

            
            scored_chunks = []
            name_parts = [p.lower() for p in char_name.split() if len(p) >= 3]
            
            for i, chunk in enumerate(chunks):
                chunk_lower = chunk.lower()
                name_score = 0.5 if any(part in chunk_lower for part in name_parts) else 0.0
                
                keyword_score = 0.0
                keywords = ['born', 'died', 'lived', 'married', 'father', 'mother', 'killed', 
                           'orphan', 'family', 'child', 'years old', 'home']
                for kw in keywords:
                    if kw in chunk_lower:
                        keyword_score += 0.05
                keyword_score = min(keyword_score, 0.3)
                
                position_score = 0.2 * (1 - i / len(chunks))
                total_score = name_score + keyword_score + position_score
                scored_chunks.append((total_score, i, chunk))
            
            scored_chunks.sort(reverse=True, key=lambda x: x[0])
            top_chunks = sorted(scored_chunks[:max_chunks], key=lambda x: x[1])
            relevant_chunks = [chunk for _, _, chunk in top_chunks]
            
            if len(relevant_chunks) < 10:
                relevant_chunks = chunks[:100]
            
            encoder.reset()
            trajectory = []
            
            # Optimized: Batch process all chunks -> Get all states
            all_states = encoder.process_chunks_in_batch(relevant_chunks, device=device)
            
            snapshot_interval = max(1, len(relevant_chunks) // num_snapshots)
            
            for i, state in enumerate(all_states):
                if i % snapshot_interval == 0 or i == len(relevant_chunks) - 1:
                    position = i / len(relevant_chunks)
                    trajectory.append({
                        'position': position,
                        'state': state.detach().cpu(), # Already cloned in process_chunks_in_batch
                        'beliefs': encoder.bdh_state.get_beliefs_summary().copy()
                    })
            
            book_char_trajectories[key] = trajectory
            
            if len(book_char_trajectories) % 5 == 0:
                print(f"  Processed {len(book_char_trajectories)}/{len(unique_pairs)} characters")
                
        except Exception as e:
            print(f"  Error: {book_name} - {e}")
            book_char_trajectories[key] = None
    
    print(f"✅ Computed {len(book_char_trajectories)} trajectories with {num_snapshots} snapshots each")
    return book_char_trajectories

from torch.utils.data import Dataset, DataLoader

class TrajectoryDataset(Dataset):
    def __init__(self, df, trajectories, encoder):
        self.df = df.reset_index(drop=True)
        self.trajectories = trajectories
        self.encoder = encoder
        self.data = []
        
        print("  Pre-encoding backstories with trajectories...")
        for idx, row in self.df.iterrows():
            key = (row['book_name'], row['char'])
            trajectory = self.trajectories.get(key)
            
            if trajectory is None:
                continue
            
            early_states = [t['state'].detach() for t in trajectory if t['position'] < 0.3]
            late_states = [t['state'].detach() for t in trajectory if t['position'] > 0.7]
            
            if early_states:
                early_state = torch.stack(early_states).mean(dim=0).detach()
            else:
                early_state = torch.zeros(256)
            
            if late_states:
                late_state = torch.stack(late_states).mean(dim=0).detach()
            else:
                late_state = torch.zeros(256)
            
            backstory_emb = self.encoder.encode_text(row['content']).detach()
            univ_features = torch.tensor(
                universal_extractor.get_feature_vector(row['content'], []),
                dtype=torch.float32
            )
            label = 1.0 if row['label'] == 'consistent' else 0.0
            
            self.data.append({
                'backstory_emb': backstory_emb,
                'early_state': early_state,
                'late_state': late_state,
                'univ_features': univ_features,
                'label': label
            })
        
        print(f"  ✅ Pre-processed {len(self.data)} samples with trajectories")
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

def collate_trajectory_fn(batch):
    return {
        'backstory_emb': torch.stack([b['backstory_emb'] for b in batch]),
        'early_state': torch.stack([b['early_state'] for b in batch]),
        'late_state': torch.stack([b['late_state'] for b in batch]),
        'univ_features': torch.stack([b['univ_features'] for b in batch]),
        'label': torch.tensor([b['label'] for b in batch])
    }

BackstoryDataset = TrajectoryDataset
collate_fn = collate_trajectory_fn

def train_classifier_only(encoder, classifier, train_df, val_df, trajectories, epochs=25, batch_size=8):
    print("\n🎓 PHASE 3: Training CAUSAL classifier (trajectory-based)...")
    
    train_dataset = TrajectoryDataset(train_df, trajectories, encoder)
    val_dataset = TrajectoryDataset(val_df, trajectories, encoder)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_trajectory_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_trajectory_fn)
    
    classifier = classifier.to(device)
    
    n_consistent = (train_df['label'] == 'consistent').sum()
    n_contradict = (train_df['label'] == 'contradict').sum()
    weight_ratio = max(n_consistent / max(n_contradict, 1), 1.0)
    pos_weight = torch.tensor([weight_ratio], device=device)
    print(f"  Class weights: Consistent={n_consistent}, Contradict={n_contradict}, pos_weight={weight_ratio:.2f}")
    
    optimizer = optim.AdamW([
        {'params': classifier.parameters(), 'lr': 1e-4},
        {'params': encoder.projection.parameters(), 'lr': 1e-5},
        {'params': encoder.bdh_state.parameters(), 'lr': 2e-4}
    ], weight_decay=0.01)
    
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.7)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    best_val_acc = 0
    patience = 10
    no_improve_count = 0
    
    for epoch in range(epochs):
        classifier.train()
        encoder.projection.train()
        encoder.bdh_state.train()
        
        total_loss = 0
        correct = 0
        total = 0
        
        for batch in train_loader:
            backstory_embs = batch['backstory_emb'].to(device)
            early_states = batch['early_state'].to(device)
            late_states = batch['late_state'].to(device)
            univ_features = batch['univ_features'].to(device)
            labels = batch['label'].to(device)
            
            trajectory_tuple = (early_states, late_states)
            preds = classifier(backstory_embs, trajectory_tuple, univ_features)
            loss = criterion(preds.view(-1), labels)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item() * len(labels)
            pred_labels = (torch.sigmoid(preds) > 0.5).long()
            correct += (pred_labels.view(-1) == labels.long()).sum().item()
            total += len(labels)
        
        classifier.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for batch in val_loader:
                backstory_embs = batch['backstory_emb'].to(device)
                early_states = batch['early_state'].to(device)
                late_states = batch['late_state'].to(device)
                univ_features = batch['univ_features'].to(device)
                labels = batch['label'].to(device)
                
                trajectory_tuple = (early_states, late_states)
                preds = classifier(backstory_embs, trajectory_tuple, univ_features)
                pred_labels = (torch.sigmoid(preds) > 0.5).long()
                val_correct += (pred_labels.view(-1) == labels.long()).sum().item()
                val_total += len(labels)
        
        val_acc = val_correct / val_total if val_total > 0 else 0
        scheduler.step()
        
        train_acc = correct / total if total > 0 else 0
        avg_loss = total_loss / total if total > 0 else 0
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Train: {train_acc:.2%} | Val: {val_acc:.2%}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve_count = 0
            torch.save({
                'projection': encoder.projection.state_dict(),
                'bdh_state': encoder.bdh_state.state_dict(),
                'classifier': classifier.state_dict(),
            }, f'{OUTPUT_DIR}/best_causal_model.pt')
            print(f"  → Saved best model ({best_val_acc:.2%})")
        else:
            no_improve_count += 1
            if no_improve_count >= patience:
                print(f"⏹️ Early Stopping! Best: {best_val_acc:.2%}")
                break
    
    return classifier
def evaluate_with_details(encoder, classifier, df, book_states):
    classifier.eval()
    correct = 0
    total = 0
    predictions = []
    with torch.no_grad():
        for _, row in df.iterrows():
            key = (row['book_name'], row['char'])
            story_state = book_states.get(key)
            if story_state is None: continue
            story_state = story_state.to(device)
            back_emb = encoder.encode_text(row['content']).to(device)
            univ_features = torch.tensor(
                universal_extractor.get_feature_vector(row['content'], []),
                device=device
            )
            pred = classifier(story_state, back_emb, univ_features)
            confidence = torch.sigmoid(pred).item()
            pred_label = 1 if confidence > 0.5 else 0
            label = 1 if row['label']=='consistent' else 0
            correct += (pred_label == label)
            total += 1
            predictions.append({'confidence': confidence, 'pred': pred_label, 'true': label})
    return correct / total if total > 0 else 0, predictions
novel_chunks_cache = {}
novel_embeddings_cache = {}
# Removed dead code: unused prediction functions

def generate_causal_predictions(encoder, classifier, test_df, trajectories, books_dir):
    classifier.eval()
    encoder.eval()
    results = []
    confidences = []
    
    print("\n📝 Generating predictions with CAUSAL TRAJECTORY ANALYSIS...")
    
    for idx, row in test_df.iterrows():
        key = (row['book_name'], row['char'])
        backstory = row['content']
        trajectory = trajectories.get(key)
        # OPTIMIZATION: Ensure cache is populated FIRST so fallback functions can use it
        if row['book_name'] not in novel_chunks_cache:
            novel_text = load_novel(row['book_name'], books_dir)
            if novel_text:
                full_chunks = chunk_novel(novel_text)
                if len(full_chunks) > 1500: full_chunks = full_chunks[:1500]  # OOM safety
                
                novel_chunks_cache[row['book_name']] = full_chunks
                with torch.no_grad():
                    novel_embeddings_cache[row['book_name']] = encoder.encoder.encode(
                        full_chunks, convert_to_tensor=True, device=device, batch_size=32
                    )
                print(f"  📚 Cached book '{row['book_name']}': {len(full_chunks)} chunks")
            else:
                novel_chunks_cache[row['book_name']] = []
                novel_embeddings_cache[row['book_name']] = None

        full_chunks = novel_chunks_cache.get(row['book_name'], [])
        full_embeddings = novel_embeddings_cache.get(row['book_name'])
        
        # Now compute trajectory if missing (using CACHED embeddings for speed!)
        if trajectory is None or len(trajectory) == 0:
            try:
                if full_chunks:  # Only if book exists
                    # Efficient selection using cache
                    relevant = get_backstory_relevant_chunks(
                        backstory, full_chunks, encoder, top_k=200, cached_embeddings=full_embeddings
                    )
                    if len(relevant) < 5:
                        relevant = full_chunks[:200]
                    
                    encoder.reset()
                    trajectory = []
                    
                    # Optimized: Batch process
                    all_states = encoder.process_chunks_in_batch(relevant, device=device)
                    
                    snapshot_interval = max(1, len(relevant) // 10)
                    
                    for i, state in enumerate(all_states):
                        if i % snapshot_interval == 0 or i == len(relevant) - 1:
                            trajectory.append({
                                'position': i / len(relevant),
                                'state': state.detach().cpu(),
                                'beliefs': encoder.bdh_state.get_beliefs_summary().copy()
                            })
            except Exception as e:
                print(f"  ⚠️ Error computing trajectory for {key}: {e}")
                trajectory = None
        
        if trajectory is None or len(trajectory) == 0:
            results.append({'id': row['id'], 'label': 1, 'rationale': 'No data available - default consistent'})
            confidences.append(0.5)
            continue
        
        # 3. Compute evidence chunks using cached embeddings
        evidence_chunks = find_evidence_chunks(backstory, full_chunks, full_embeddings, encoder, top_k=3)
        
        # Optimization: Reuse embedding if possible, but simplicity is key here. 
        # Actually, let's just encode it once.
        backstory_emb = encoder.encode_text(backstory).to(device)
        
        univ_features = torch.tensor(
            universal_extractor.get_feature_vector(backstory, evidence_chunks),
            device=device
        )
        
        with torch.no_grad():
            output = classifier(backstory_emb.unsqueeze(0), trajectory, univ_features.unsqueeze(0))
            nn_conf = torch.sigmoid(output).item()
        
        nn_pred = 1 if nn_conf > 0.5 else 0
        
        rule_vote, rule_reasons = rule_based_check(backstory, evidence_chunks)
        if nn_pred == rule_vote:
            final_pred = nn_pred
        elif rule_vote is not None:
            if nn_conf < 0.55:
                final_pred = rule_vote
            else:
                final_pred = nn_pred
        else:
            final_pred = nn_pred
        
        try:
            early_beliefs = trajectory[1]['beliefs'] if len(trajectory) > 1 else {}
            late_beliefs = trajectory[-1]['beliefs'] if trajectory else {}
            belief_changes = [k for k in early_beliefs if k in late_beliefs and early_beliefs[k].get('value') != late_beliefs[k].get('value')]
            
            if belief_changes:
                causal_note = f"Causal path: {len(belief_changes)} belief evolution(s) detected."
            else:
                causal_note = "Stable causal path from backstory to outcome."
        except:
            causal_note = ""
        
        rationale = generate_rationale(
            backstory, evidence_chunks,
            nn_pred, nn_conf,
            rule_vote, rule_reasons,
            final_pred
        )
        if causal_note:
            rationale = causal_note + " " + rationale
        
        results.append({'id': row['id'], 'label': final_pred, 'rationale': rationale})
        confidences.append(nn_conf)
        
        if (idx + 1) % 10 == 0:
            print(f"  Processed {idx + 1}/{len(test_df)} predictions...")
    
    results_df = pd.DataFrame(results)
    print(f"\n📊 Prediction Distribution:")
    print(results_df['label'].value_counts())
    print(f"\n📈 Confidence Stats:")
    print(f"  Mean: {np.mean(confidences):.3f}")
    print(f"  Std:  {np.std(confidences):.3f}")
    print(f"  Min:  {np.min(confidences):.3f}")
    print(f"  Max:  {np.max(confidences):.3f}")
    print(f"\n{'='*60}")
    print("📜 SAMPLE RATIONALES (First 3):")
    print('='*60)
    for i, row in results_df.head(3).iterrows():
        print(f"\n--- ID: {row['id']} ---")
        print(row['rationale'])
    
    return results_df
if __name__ == "__main__":
    STATE_DIM = 256
    encoder = ChunkEncoder(state_dim=STATE_DIM).to(device)
    
    print("\n" + "="*60)
    print("🚀 CAUSAL PATH ARCHITECTURE - BDH Track B Implementation")
    print("="*60)
    
    print("\n📊 PHASE 1: Pre-computing TRAJECTORIES for training data only...")
    train_trajectories = precompute_trajectories(train_df, encoder, BOOKS_DIR, max_chunks=200, num_snapshots=10)
    print(f"✅ Computed {len(train_trajectories)} training trajectories (NO test data leakage)")
    
    print("\nSplitting data...")
    train_split, val_split = train_test_split(train_df, test_size=0.2, random_state=42)
    print("Augmenting training set only...")
    train_data = create_augmented_data(train_split)
    val_data = val_split
    print(f"Final sizes -> Train: {len(train_data)} (augmented), Val: {len(val_data)} (pure)")
    
    classifier = CausalCompatibilityClassifier(state_dim=STATE_DIM)
    classifier = train_classifier_only(encoder, classifier, train_data, val_data, train_trajectories, epochs=30)  # FIX 3: Reduced from 50 to 30
    
    checkpoint = torch.load(f'{OUTPUT_DIR}/best_causal_model.pt')
    encoder.projection.load_state_dict(checkpoint['projection'])
    encoder.bdh_state.load_state_dict(checkpoint['bdh_state'])
    classifier.load_state_dict(checkpoint['classifier'])
    
    print("\n🔄 PHASE 2: Computing TEST trajectories with TRAINED encoder...")
    
    # FIX 1: REMOVED redundant train trajectory recomputation - saves ~30-60 min
    # Training trajectories capture narrative structure which is stable
    
    print("  → Computing test trajectories (first time, with trained encoder)...")
    test_trajectories = precompute_trajectories(test_df, encoder, BOOKS_DIR)
    
    all_trajectories = {**train_trajectories, **test_trajectories}  # Use original train_trajectories
    print(f"✅ Total trajectories: {len(all_trajectories)}")
    
    results_df = generate_causal_predictions(encoder, classifier, test_df, all_trajectories, BOOKS_DIR)
    # User requested removing rationale column from results.csv
    results_df[['id', 'label']].to_csv(f'{OUTPUT_DIR}/results.csv', index=False)
    results_df[['id', 'label']].to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)
    print(f"\n✅ Saved {len(results_df)} predictions to results.csv and submission.csv (Rationales excluded from file)")
    print(results_df.head(15))
    
    print("\n📊 BDH METRICS FOR REPORT:")
    print(f"  Sparsity Stats: {encoder.bdh_state.get_sparsity_stats()}")
    print(f"  Belief Summary: {encoder.bdh_state.get_beliefs_summary()}")
    
    print("\n" + "="*60)
    print("📊 GENERATING VISUALIZATIONS FOR REPORT")
    print("="*60)
    
    try:
        import matplotlib.pyplot as plt
        
        sample_row = test_df.iloc[0]
        sample_novel = load_novel(sample_row['book_name'], BOOKS_DIR)
        sample_chunks = chunk_novel(sample_novel)[:100]
        
        encoder.reset()
        
        belief_trajectory = []
        update_matrix = []
        prev_state = None
        
        backstory_emb = encoder.encode_text(sample_row['content']).to(device)
        
        for i, chunk in enumerate(sample_chunks):
            if prev_state is not None:
                delta = torch.abs(encoder.bdh_state.state - prev_state).cpu().numpy()
                update_matrix.append(delta)
            
            state = encoder.process_chunk(chunk)
            prev_state = state.clone()
            
            with torch.no_grad():
                dummy_trajectory = (state.unsqueeze(0).to(device), state.unsqueeze(0).to(device))
                output = classifier(backstory_emb.unsqueeze(0), dummy_trajectory, torch.zeros(1, 12, device=device))
                belief = torch.sigmoid(output).item()
                belief_trajectory.append(belief)
        
        update_matrix = np.array(update_matrix)
        plt.figure(figsize=(12, 6))
        plt.imshow(update_matrix.T, aspect='auto', cmap='hot', interpolation='nearest')
        plt.colorbar(label='Update Magnitude')
        plt.title('BDH Principle: Sparse Selective Updates\\n(Only 15% of dimensions change per chunk)', fontsize=14)
        plt.xlabel('Chunk Index (Reading Progress)', fontsize=12)
        plt.ylabel('State Dimension', fontsize=12)
        sparsity_rate = (update_matrix < 0.01).mean()
        plt.text(0.02, 0.98, f'Sparsity: {sparsity_rate*100:.1f}% unchanged',
                 transform=plt.gca().transAxes, fontsize=10, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/figure1_sparsity.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✅ Saved: figure1_sparsity.png")
        
        plt.figure(figsize=(10, 6))
        plt.plot(belief_trajectory, linewidth=2.5, color='#2E86AB', marker='o', markersize=3)
        plt.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Decision Threshold', alpha=0.7)
        plt.fill_between(range(len(belief_trajectory)), 0.5, 1.0, alpha=0.1, color='green')
        plt.fill_between(range(len(belief_trajectory)), 0, 0.5, alpha=0.1, color='red')
        plt.xlabel('Chunks Read (Reading Progress)', fontsize=12)
        plt.ylabel('Consistency Belief', fontsize=12)
        plt.title('BDH Principle: Incremental Belief Formation\\n(Gradual accumulation, not sudden jumps)', fontsize=14)
        plt.legend(fontsize=10)
        plt.grid(True, alpha=0.3, linestyle=':')
        plt.ylim(0, 1)
        final_decision = "Consistent" if belief_trajectory[-1] > 0.5 else "Inconsistent"
        plt.text(0.98, 0.02, f'Final: {final_decision} ({belief_trajectory[-1]:.2f})',
                 transform=plt.gca().transAxes, fontsize=10, horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/figure2_belief_evolution.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✅ Saved: figure2_belief_evolution.png")
        
        update_counts = (update_matrix > 0.01).sum(axis=0)
        plt.figure(figsize=(12, 4))
        plt.bar(range(256), update_counts, color='steelblue', alpha=0.7)
        plt.xlabel('State Dimension', fontsize=12)
        plt.ylabel('Times Updated', fontsize=12)
        active_dims = (update_counts > 0).sum()
        plt.title(f'BDH Principle: Selective Updates\\n(Only {active_dims}/256 dimensions active)', fontsize=14)
        plt.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig(f'{OUTPUT_DIR}/figure3_dimension_activity.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✅ Saved: figure3_dimension_activity.png")
        
        print(f"\n📈 REPORT STATISTICS:")
        print(f"  Sparsity Rate: {sparsity_rate*100:.1f}%")
        print(f"  Active Dimensions: {active_dims}/256")
        print(f"  Final Belief: {belief_trajectory[-1]:.3f}")
        
    except Exception as e:
        print(f"  ⚠️ Visualization error (matplotlib may not be available): {e}")
    
    print("\n" + "="*60)
    print("✅ ALL DONE! Check /kaggle/working/ for outputs:")
    print("  - results.csv (predictions)")
    print("  - figure1_sparsity.png")
    print("  - figure2_belief_evolution.png") 
    print("  - figure3_dimension_activity.png")
    print("="*60)

Using device: cuda
Random seed set to 42 for reproducibility
Train: 80, Test: 60

🚀 CAUSAL PATH ARCHITECTURE - BDH Track B Implementation

📊 PHASE 1: Pre-computing TRAJECTORIES for training data only...

🎬 PHASE 2: Pre-computing STATE TRAJECTORIES (for causal analysis)...
Found 6 unique (book, character) pairs
  ⚡ Building book filename cache...
  Processed 5/6 characters
✅ Computed 6 trajectories with 10 snapshots each
✅ Computed 6 training trajectories (NO test data leakage)

Splitting data...
Augmenting training set only...
🔄 AUGMENTATION COMPLETE (2x safer):
   Original: 64 samples
   Added: 1 samples
   Total: 65 samples (1.0x increase)
Final sizes -> Train: 65 (augmented), Val: 16 (pure)

🎓 PHASE 3: Training CAUSAL classifier (trajectory-based)...
  Pre-encoding backstories with trajectories...
  ✅ Pre-processed 65 samples with trajectories
  Pre-encoding backstories with trajectories...
  ✅ Pre-processed 16 samples with trajectories
  Class weights: Consistent=40, Contradict=25,